In [1]:

import numpy as np
import random

# 1. Setting up the environment
class TicTacToe:
    def __init__(self):
        self.board = [' '] * 9

    def reset(self):
        self.board = [' '] * 9
        return self.get_state()

    def get_state(self):
        return ''.join(self.board)

    def available_actions(self):
        return [i for i in range(9) if self.board[i] == ' ']

    def make_move(self, action, player):
        if self.board[action] == ' ':
            self.board[action] = player
            return True
        return False

    def check_winner(self):
        wins = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],
            [0, 3, 6], [1, 4, 7], [2, 5, 8],
            [0, 4, 8], [2, 4, 6]
        ]

        for a, b, c in wins:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]

        if ' ' not in self.board:
            return 'Draw'

        return None

    def display(self):
        for i in range(0, 9, 3):
            print(' | '.join(self.board[i:i+3]))
            if i < 6:
                print('---------')




In [2]:
# 2. Building the reinforcement learning model
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q_values(self, state):
        if state not in self.q_table:
            self.q_table[state] = np.zeros(9)
        return self.q_table[state]

    def choose_action(self, state, actions):
        if random.random() < self.epsilon:
            return random.choice(actions)

        q_values = self.get_q_values(state)
        return max(actions, key=lambda a: q_values[a])

    def learn(self, state, action, reward, next_state, done):
        q_values = self.get_q_values(state)

        if done:
            target = reward
        else:
            next_actions = [
                i for i, cell in enumerate(next_state) if cell == ' '
            ]

            if next_actions:
                target = reward + self.gamma * max(
                    self.get_q_values(next_state)[a]
                    for a in next_actions
                )
            else:
                target = reward

        q_values[action] += self.alpha * (
            target - q_values[action]
        )




In [3]:
# 3. Training the model
env = TicTacToe()
agent = QLearningAgent()

episodes = 30000

for episode in range(episodes):
    state = env.reset()
    done = False

    while not done:

        # Agent plays as X
        actions = env.available_actions()
        action = agent.choose_action(state, actions)
        env.make_move(action, 'X')

        winner = env.check_winner()

        if winner == 'X':
            agent.learn(state, action, 1, env.get_state(), True)
            break

        if winner == 'Draw':
            agent.learn(state, action, 0, env.get_state(), True)
            break

        # Opponent plays randomly as O
        opponent_action = random.choice(env.available_actions())
        env.make_move(opponent_action, 'O')

        winner = env.check_winner()

        if winner == 'O':
            agent.learn(state, action, -1, env.get_state(), True)
            break

        if winner == 'Draw':
            agent.learn(state, action, 0, env.get_state(), True)
            break

        next_state = env.get_state()

        agent.learn(state, action, 0, next_state, False)

        state = next_state

    agent.epsilon = max(0.05, agent.epsilon * 0.9999)

print("Training completed successfully!")
print("Total Episodes:", episodes)
print("Q-Table Size:", len(agent.q_table))




Training completed successfully!
Total Episodes: 30000
Q-Table Size: 2394


In [4]:
# 4. Testing the model
def test_agent():
    env = TicTacToe()
    agent.epsilon = 0

    print("\nTIC TAC TOE GAME")
    print("You are O and AI is X")
    print("Enter positions from 0 to 8")

    print("\nPosition Guide:")
    print("0 | 1 | 2")
    print("---------")
    print("3 | 4 | 5")
    print("---------")
    print("6 | 7 | 8")

    state = env.reset()

    while True:

        # AI move
        actions = env.available_actions()
        action = agent.choose_action(state, actions)
        env.make_move(action, 'X')

        print("\nAI Move:")
        env.display()

        winner = env.check_winner()

        if winner:
            print("Result:", "AI Wins!" if winner == 'X' else "Draw!")
            break

        # Human move
        while True:
            try:
                human_action = int(input("\nEnter your move (0-8): "))

                if human_action in env.available_actions():
                    break

                print("Invalid position! Try again.")

            except ValueError:
                print("Please enter a number between 0 and 8.")

        env.make_move(human_action, 'O')

        print("\nYour Move:")
        env.display()

        winner = env.check_winner()

        if winner:
            print("Result:", "You Win!" if winner == 'O' else "Draw!")
            break




In [5]:
# 5. Run the game
test_agent()



TIC TAC TOE GAME
You are O and AI is X
Enter positions from 0 to 8

Position Guide:
0 | 1 | 2
---------
3 | 4 | 5
---------
6 | 7 | 8

AI Move:
  |   | X
---------
  |   |  
---------
  |   |  

Your Move:
O |   | X
---------
  |   |  
---------
  |   |  

AI Move:
O | X | X
---------
  |   |  
---------
  |   |  

Your Move:
O | X | X
---------
  |   |  
---------
  |   | O

AI Move:
O | X | X
---------
  |   | X
---------
  |   | O

Your Move:
O | X | X
---------
  | O | X
---------
  |   | O
Result: You Win!
